# Step 0 · Validation of the SBERT filter on PROD

On TEST the filter was validated in `1-DataPreparation` (precision 0.88, recall 0.945). TEST is a sample of PATSTAT, so here the same check is repeated on the **PROD** run, the one used for the final dataset.

- **Input:** `df_sbert_scored_PROD.parquet` (all PROD applications with an English abstract and their SBERT similarity; SBERT run on Colab with the same model, description and threshold; identical abstracts encoded once). The kept/rejected split is rebuilt here from the threshold.
- **Method:** the same as on TEST (`2-DataValidation`): random families from the kept and from the rejected group (seed 42), one abstract per family, labelled by gpt-4.1-mini with the same prompt; `incerto` answers are resolved by hand; precision and recall with 95% Wilson intervals.
- **Cost:** about 5 cents per 100 LLM calls; families already labelled are read from the cache.
- **Output:** `validation/prod_filter_validation.csv` (numbers only). The review files (`review_filtro_sbert_prod_*.csv`) contain titles and abstracts: do not commit them.

In [ ]:
from pathlib import Path
import re

import pandas as pd
from IPython.display import display

SCORED_FILE = Path("/home/jovyan/progetto/prod/df_sbert_scored_PROD.parquet")  # Soroush's PROD SBERT output

TAU_FILTER = 0.45                # same threshold as 1-DataPreparation
N_KEPT = 100                     # random families checked among the kept ones
N_REJECTED = 100                 # and among the rejected ones (same size as on TEST)
RUN_NAME = "Filtro SBERT PROD"   # review files: validation/review_filtro_sbert_prod_*.csv

# Must be identical to 1-DataPreparation: the LLM prompt and the cache depend on it.
DRONE_DESCRIPTION = """
Unmanned aerial vehicles, remotely piloted aircraft and autonomous flying
drones. Technologies specifically designed for their airframe, propulsion,
flight control, navigation, communications, swarm coordination, dedicated
charging infrastructure, launch, landing, recovery and mission equipment.
The invention must provide a concrete technical solution for an unmanned
aircraft. Generic technologies with only an incidental drone application,
ground-based anti-drone systems, manned aircraft and exclusively terrestrial
or underwater vehicles are outside the sector.
""".strip()

## 1. Kept and rejected families

As in `1-DataPreparation` (`split_validation_families`): a family is **kept** if at least one of its applications reaches the threshold; it is **rejected** if none does.

In [ ]:
scored = pd.read_parquet(SCORED_FILE)
scored = scored.dropna(subset=["docdb_family_id"]).copy()
scored["docdb_family_id"] = scored["docdb_family_id"].astype("int64")
scored = scored[scored["docdb_family_id"] > 0]

kept = scored[scored["sbert_similarity"] >= TAU_FILTER].copy()
kept_ids = set(kept["docdb_family_id"])
rejected = scored[(scored["sbert_similarity"] < TAU_FILTER)
                  & ~scored["docdb_family_id"].isin(kept_ids)].copy()

assert kept_ids.isdisjoint(set(rejected["docdb_family_id"]))
print(f"Applications scored: {len(scored):,} | kept: {len(kept):,} ({len(kept) / len(scored):.1%})")
print(f"Families kept: {kept['docdb_family_id'].nunique():,} | rejected: {rejected['docdb_family_id'].nunique():,}")

## 2. LLM labels

Loads the validator and checks that the prompt is the one used on TEST (otherwise the two results would not be comparable).

In [ ]:
%pip install -q langchain-core langchain-openai pydantic
%run -i ./2-DataValidation.ipynb

pid = prompt_id(DRONE_DESCRIPTION)
test_prompts = {r["prompt_id"] for r in load_llm_cache().values() if r.get("prompt_name") == PROMPT_NAME}
assert pid in test_prompts, "Prompt differs from the TEST run: check DRONE_DESCRIPTION"
print("Same prompt as on TEST:", pid)

validation_kept = validate_sample(kept, n=N_KEPT)
validation_rejected = validate_sample(rejected, n=N_REJECTED)

prod_metrics = evaluate_selection(
    validation_selected=validation_kept,
    validation_rejected=validation_rejected,
    df_selected=kept,
    df_rejected=rejected,
    name=RUN_NAME,
)
display(prod_metrics.to_frame().round(3))

## 3. Results

If `incerto` cases are listed above, label them in the review files (`human_label` = `pertinente` or `non_pertinente`) and re-run the notebook: the LLM is not called again.

- **precision:** share of the kept families that are drones; **estimated drone families** = kept families × precision.
- **relevant_share_in_rejected:** share of drones among the rejected families; **estimated drone families lost** = rejected families × that share.
- **recall_within_pool:** drones kept / drones found by the search (the search itself can miss drones: this is recall *within* the search results).

###### if prod_metrics.get("to_check_by_hand", 0):
    print("Label the 'incerto' cases first, then re-run.")
else:
    summary = prod_metrics.copy()
    summary["estimated_drone_families_kept"] = summary["selected_families"] * summary["precision"]
    summary["estimated_drone_families_lost"] = (summary["rejected_families"]
                                                * summary["relevant_share_in_rejected"])
    summary.to_frame().T.round(4).to_csv(VALIDATION_DIR / "prod_filter_validation.csv", index=False)
    display(summary.to_frame().round(3))
    print("Saved:", VALIDATION_DIR / "prod_filter_validation.csv")

## 4. The errors

Which families are wrong: non-drones among the kept ones (they lower precision) and drones among the rejected ones (they lower recall). Shown here only (titles are not saved), to describe the errors in the appendix.

In [ ]:
slug = re.sub(r"[^a-z0-9]+", "_", RUN_NAME.lower()).strip("_")
for title, sample, tag, wrong in [
    ("Kept but not drones", validation_kept, "selezionati", "non_pertinente"),
    ("Rejected but drones", validation_rejected, "scartati", "pertinente"),
]:
    labels = _resolved_labels(sample, VALIDATION_DIR / f"review_{slug}_{tag}.csv")
    errors = sample.reset_index(drop=True)[labels.eq(wrong)]
    print(f"\n{title}: {len(errors)} of {len(sample)}")
    display(errors[["docdb_family_id", "sbert_similarity", "appln_title", "reason"]])